### Modelisation

Model creation to predict the missing data

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# ========== 1. DATA LOADING ==========
print("Loading datasets...")
# Load the real training and testing datasets
df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

# ========== 2. DATA PREPROCESSING (Aggregation) ==========

def aggregate_to_weekly(df, is_train=True):
    """
    Groups daily temporal data into weekly intervals.
    - Precipitation is summed over the week.
    - Temperatures, humidity, and wind are averaged.
    - Score takes the maximum value (to capture the single non-NaN value).
    """
    # Create a 'week' index for every 7 days, per region
    df['week'] = df.groupby('region_id').cumcount() // 7
    
    # Define how each feature should be aggregated
    # Using the real column names provided
    agg_dict = {
        'prec': 'sum',           # Total rain over the week
        'humidity': 'mean',      # Average humidity
        'tmp': 'mean',           # Average temperature
        'tmp_max': 'max',        # Highest temperature of the week
        'tmp_min': 'min',        # Lowest temperature of the week
        'wind': 'mean',          # Average wind
        'wind_max': 'max'        # Maximum wind gust of the week
    }
    
    if is_train:
        agg_dict['score'] = 'max' # Captures the recorded score, ignores NaNs
        
    # Apply the aggregation
    df_weekly = df.groupby(['region_id', 'week']).agg(agg_dict).reset_index()
    return df_weekly

def generate_future_targets(df_weekly):
    """
    Shifts the target variable backwards to map current week's weather
    to the drought scores of the next 5 weeks (t+1 to t+5).
    """
    for i in range(1, 6):
        # Shift creates the target for week 1, week 2, etc.
        df_weekly[f'target_week_{i}'] = df_weekly.groupby('region_id')['score'].shift(-i)
        
    # Drop rows that now lack future targets (the very last weeks of the dataset)
    df_weekly = df_weekly.dropna()
    return df_weekly

print("Aggregating daily data to weekly datasets...")
train_weekly = aggregate_to_weekly(df_train, is_train=True)
test_weekly = aggregate_to_weekly(df_test, is_train=False)

print("Creating forecasting targets...")
train_final = generate_future_targets(train_weekly)

# ========== 3. MODEL TRAINING==========

# Define features used for prediction
features = ['prec', 'humidity', 'tmp', 'tmp_max', 'tmp_min', 'wind', 'wind_max']
target_cols = ['target_week_1', 'target_week_2', 'target_week_3', 'target_week_4', 'target_week_5']

X_train = train_final[features]
y_train = train_final[target_cols]

print("Training the RandomForestRegressor with constrained hyperparameters...")

# We adjust the hyperparameters to prevent memory overload and speed up training:
# - n_estimators: Reduced to 20 trees (faster to compute than 100)
# - max_depth: Capped at 5 to prevent trees from growing too deep and consuming all RAM
# - min_samples_split: Requires at least 10 samples to split a node, reducing complexity
model = RandomForestRegressor(
    n_estimators=20, 
    max_depth=5, 
    min_samples_split=10,
    random_state=42, 
    n_jobs=-1 # Uses all CPU cores available
)

model.fit(X_train, y_train)
print("Model training complete!")

# ========== 4. PREDICTION & SUBMISSION ==========

print("Predicting the next 5 weeks for the test set...")
# Isolate the very last week of test data (representing the most recent conditions)
test_last_week = test_weekly.groupby('region_id').tail(1)
X_test = test_last_week[features]

# Generate predictions
predictions = model.predict(X_test)

# Format the submission file exactly as required by the competition
submission = pd.DataFrame(predictions, columns=['pred_week1', 'pred_week2', 'pred_week3', 'pred_week4', 'pred_week5'])
submission.insert(0, 'region_id', test_last_week['region_id'].values)

submission.to_csv('submission.csv', index=False)
print("Pipeline complete! 'submission.csv' generated successfully.")

Loading datasets...
Aggregating daily data to weekly datasets...
Creating forecasting targets...
Training the RandomForestRegressor with constrained hyperparameters...
Model training complete!
Predicting the next 5 weeks for the test set...
Pipeline complete! 'submission.csv' generated successfully.
